<a href="https://colab.research.google.com/github/ochilovu2010/IOAI/blob/main/Topics/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [254]:
!wget https://raw.githubusercontent.com/artemovae/NLP-seminar-LM/master/dinos.txt

--2026-06-09 10:31:52--  https://raw.githubusercontent.com/artemovae/NLP-seminar-LM/master/dinos.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 19909 (19K) [text/plain]
Saving to: ‘dinos.txt.14’

dinos.txt.14        100%[===================>]  19.44K  --.-KB/s    in 0.001s  

2026-06-09 10:31:52 (13.6 MB/s) - ‘dinos.txt.14’ saved [19909/19909]



In [255]:
!cat dinos.txt | head

Aachenosaurus
Aardonyx
Abdallahsaurus
Abelisaurus
Abrictosaurus
Abrosaurus
Abydosaurus
Acanthopholis
Achelousaurus
Acheroraptor


In [256]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from torch.nn.utils.rnn import pad_sequence

In [257]:
class dinosdataset(Dataset):
  def __init__(self):
    super().__init__()
    with open('dinos.txt') as f:
      content = f.read().lower()
      self.vocab = sorted(set(content)) + ['.']
      self.lines = content.splitlines()
      self.vocab_size = len(self.vocab)
    self.ch_to_idx = {x:i for i, x in enumerate(self.vocab)}
    self.idx_to_ch = {i:x for i, x in enumerate(self.vocab)}
    self.padding_idx = self.ch_to_idx['.']
  def __len__(self):
    return len(self.lines)
  def __getitem__(self, x):
    name = self.lines[x] + '.'
    encoded = [self.ch_to_idx[i] for i in name]

    x = encoded[:-1]
    y = encoded[1:]
    return torch.tensor(x), torch.tensor(y)

In [258]:
data = dinosdataset()
x, y = data[1]

def custom_collate_fn(batch):
  inputs = [datas[0] for datas in batch]
  target = [datas[1] for datas in batch]
  # Use the correct padding_value from the dataset
  inputs_padded = pad_sequence(inputs, batch_first = True, padding_value=data.padding_idx)
  target_padded = pad_sequence(target, batch_first = True, padding_value=data.padding_idx)
  return inputs_padded, target_padded

loader = DataLoader(data, batch_size = 16, shuffle = True, collate_fn = custom_collate_fn)

In [259]:
print(x, y)

tensor([ 1,  1, 18,  4, 15, 14, 25, 24]) tensor([ 1, 18,  4, 15, 14, 25, 24, 27])


In [260]:
class rnnmodel(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, embedding_dim)
    self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
    self.fc = nn.Linear(hidden_dim, vocab_size)
  def forward(self, x, hidden = None):
    embedding = self.embed(x)
    out, hidden = self.rnn(embedding, hidden)
    logits = self.fc(out)
    return logits, hidden

In [261]:
model = rnnmodel(data.vocab_size, 256, 256)

In [262]:
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)
criterion = nn.CrossEntropyLoss()

In [263]:
epochs = 30
for epoch in range(epochs):
  pbar = tqdm(loader)
  running_loss = []
  for x, y in pbar:
    optimizer.zero_grad()
    logits, _ = model(x)
    logits_reshaped = logits.view(-1, logits.size(-1))
    y_reshaped = y.view(-1)
    loss = criterion(logits_reshaped, y_reshaped)
    loss.backward()
    optimizer.step()
    running_loss.append(loss.item())
  print(sum(running_loss)/len(running_loss))


100%|██████████| 96/96 [00:03<00:00, 24.76it/s]


1.4771281201392412



100%|██████████| 96/96 [00:02<00:00, 36.77it/s]


1.1516306089858215



100%|██████████| 96/96 [00:02<00:00, 36.66it/s]


...

0.36778769238541525



100%|██████████| 96/96 [00:02<00:00, 35.92it/s]


0.3610192315342526



100%|██████████| 96/96 [00:02<00:00, 35.79it/s]

0.3579068485026558


In [264]:
def sample_from_disribution(start_char):
  with torch.no_grad():
    max_length = 20
    generated = start_char
    hidden = None
    current_idx = torch.tensor([[data.ch_to_idx[generated]]])
    temperature = 0.5
    for _ in range(max_length):
      logits, hidden = model(current_idx, hidden)
      logits = logits/temperature
      probs = torch.softmax(logits.squeeze(), dim = 0)
      next_idx = torch.multinomial(probs, num_samples = 1).item()
      if next_idx == data.padding_idx:
        continue
      if next_idx == data.ch_to_idx['.']:
        break
      next_chr = data.idx_to_ch[next_idx]
      current_idx = torch.tensor([[next_idx]])
      generated += next_chr
  return generated

In [272]:
name = sample_from_disribution('c')
print(name, len(name))

crichtonpelta 13
